In [3]:
# AI4I PREDICTIVE MAINTENANCE PIPELINE  —  ENTERPRISE EDITION

!pip install -q imbalanced-learn xgboost catboost lightgbm gdown

In [4]:
#Mount Google Drive and configure the Python path

import os
import sys
import logging

from google.colab import drive

print("Mounting Google Drive...")
drive.mount("/content/drive")

PROJECT_PATH = os.environ.get(
    "PROJECT_PATH",
    "/content/drive/MyDrive/predictive-maintenance-engine",
)

if not os.path.isdir(PROJECT_PATH):
    raise FileNotFoundError(
        f"Project directory not found: '{PROJECT_PATH}'\n"
    
    )

print(f"Project path confirmed: {PROJECT_PATH}")

os.chdir(PROJECT_PATH)

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

print(f"Working directory : {os.getcwd()}")
print(f"Python path entry : {sys.path[0]}")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project path confirmed: /content/drive/MyDrive/predictive-maintenance-engine
Working directory : /content/drive/MyDrive/predictive-maintenance-engine
Python path entry : /content/drive/MyDrive/predictive-maintenance-engine


In [5]:
#  Import all project modules
try:
    from src.config import TARGET_COL, ARTIFACTS_DIR
    from src.data_ingestion import clean_data, load_data
    from src.feature_engineering import build_features_and_split, get_preprocessor
    from src.modeling import train_and_benchmark, tune_champion_model
    from src.evaluation import evaluate_and_plot, save_model

    print("All modules imported successfully.")
except ImportError as exc:
    # Raise (not sys.exit) so the Colab kernel stays alive and the traceback
    # is visible — the user can fix the issue without restarting the session.
    raise ImportError(
        f"Module import failed: {exc}\n"
        f"Make sure all dependencies are installed:\n"
        f"  !pip install -r {PROJECT_PATH}/requirements.txt"
    ) from exc

All modules imported successfully.


In [6]:
# CELL 4 — Run the full MLOps pipeline

print("\n" + "=" * 70)
print("  STARTING PREDICTIVE MAINTENANCE PIPELINE")
print("=" * 70)

# Step 1 : Data ingestion 
df_raw   = load_data()
df_clean = clean_data(df_raw)

#Step 2 : Feature engineering + three-way stratified split 
# Returns: X_train, X_val, X_test, y_train, y_val, y_test
# • X_val / y_val  → threshold optimisation ONLY  (never seen during training)
# • X_test / y_test → final unbiased evaluation   (never touched before step 6)
X_train, X_val, X_test, y_train, y_val, y_test = build_features_and_split(
    df_clean, TARGET_COL
)

# Step 3 : Build preprocessing pipeline (unfitted)
preprocessor = get_preprocessor()

# Step 4 : Benchmark model zoo (5-fold CV + SMOTE)
champion_model, champion_name, leaderboard = train_and_benchmark(
    X_train, y_train, X_test, y_test, preprocessor
)

# Step 5 : Hyperparameter tuning (business-cost objective) 
tuned_champion = tune_champion_model(
    champion_model, champion_name, X_train, y_train
)

# Step 6 : Evaluate on held-out test set + save artefacts 
# Threshold is optimised on X_val / y_val internally.
# Final metrics are reported on X_test / y_test — touched for the first time.
evaluate_and_plot(
    tuned_champion, champion_name,
    X_val, X_test,
    y_val, y_test,
)

# Step 7 : Persist production model 
save_model(tuned_champion, champion_name)

print("\n" + "=" * 70)
print(f"  PIPELINE COMPLETE")
print(f"  Artefacts saved to: {ARTIFACTS_DIR}")
print("=" * 70)


  STARTING PREDICTIVE MAINTENANCE PIPELINE



   MASTER MODEL LEADERBOARD  (ranked by 5-fold CV F1 — higher is better)
                 Model  CV_F1_Mean  CV_F1_Std  CV_AUC_Mean  Test_F1  Test_AUC  Accuracy  Precision  Recall
0             LightGBM      0.7857     0.0626       0.9707   0.7808    0.9863    0.9840     0.7308  0.8382
1             CatBoost      0.7758     0.0512       0.9709   0.7200    0.9782    0.9790     0.6585  0.7941
2              XGBoost      0.7543     0.0615       0.9638   0.7125    0.9799    0.9770     0.6196  0.8382
3        Random Forest      0.7346     0.0522       0.9698   0.7355    0.9727    0.9795     0.6552  0.8382
4    Gradient Boosting      0.6227     0.0217       0.9726   0.5957    0.9794    0.9620     0.4667  0.8235
5        Decision Tree      0.5953     0.0370       0.8653   0.6067    0.8826    0.9650     0.4909  0.7941
6           SVC (Prob)      0.4972     0.0263       0.9621   0.4917    0.9731    0.9390     0.3430  0.8676
7  Logistic Regression      0.2857     0.0147       0.9191   0.3021   